<a href="https://colab.research.google.com/github/SithmiDesilva/Voices-on-School-Reform-An-NLP-Analysis-of-Sri-Lankan-YouTube-Comments/blob/Data-collection-and-EDA/Voices_on_School_Reform_An_NLP_Analysis_of_Sri_Lankan_YouTube_Comments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports and Versions

In [1]:
import os, re, json, time, random, hashlib, math
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

SEED = 42
random.seed(SEED); np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

print('Python                  ', sys.version.split()[0])
print('pandas                  ', pd.__version__)
print('numpy                   ', np.__version__)
print('matplotlib              ', matplotlib.__version__)
print('scikit-learn            ', sklearn.__version__)
print('google-api-python-client', version('google-api-python-client'))

Python                   3.13.16
pandas                   2.2.3
numpy                    2.4.6
matplotlib               3.10.0
scikit-learn             1.6.1
google-api-python-client 2.201.0


### Mount Google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/NLP_CW')
RAW_DIR  = BASE_DIR / 'data' / 'raw'
CKPT_DIR = RAW_DIR / 'comments_by_video'      # one JSON per video (checkpoint)
FIG_DIR  = BASE_DIR / 'figures'
for d in (RAW_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(BASE_DIR)

Mounted at /content/drive
/content/drive/MyDrive/NLP_CW


### API client

In [3]:
from google.colab import userdata
YT_API_KEY = userdata.get('YT_API_KEY')
youtube = build('youtube', 'v3', developerKey=YT_API_KEY, cache_discovery=False)

try:
    AUTHOR_SALT = userdata.get('AUTHOR_SALT')
except Exception:
    AUTHOR_SALT = 'change-me-and-store-in-secrets'
print('YouTube client ready')

YouTube client ready


### Search queries and settings

In [4]:
SEARCH_QUERIES = {
    'en': [
        'Sri Lanka education reform',
        'Sri Lanka new education reforms 2026',
        'Sri Lanka school curriculum change',
        'Sri Lanka O/L A/L exam reform',
        'Sri Lanka grade 1 grade 6 new curriculum',
        'Sri Lanka education reform teachers',
        'Sri Lanka modular education system school',
        'Sri Lanka vocational education schools reform',
        'Sri Lanka digital education schools',
        'Sri Lanka history subject removed curriculum',
        'Sri Lanka education minister reform debate',
    ],
    'si': [
        'අධ්‍යාපන ප්‍රතිසංස්කරණ',
        'නව අධ්‍යාපන ප්‍රතිසංස්කරණ',
        'අධ්‍යාපන ප්‍රතිසංස්කරණ ගුරුවරු',
        'නව විෂය නිර්දේශය පාසල්',
        'සාමාන්‍ය පෙළ උසස් පෙළ වෙනස්කම්',
        '6 ශ්‍රේණිය මොඩියුල',
        'පළමු ශ්‍රේණිය නව ප්‍රතිසංස්කරණ',
        'adyapana prathisanskarana',       # romanised Sinhala ("Singlish")
    ],
}

PUBLISHED_AFTER = '2024-01-01T00:00:00Z'   # current reform cycle
REGION_CODE = 'LK'
RESULTS_PER_QUERY = 25          # max 50 per search call
MIN_COMMENT_COUNT = 15          # skip videos with very few comments
MAX_COMMENTS_PER_VIDEO = 800    # per-video ceiling (30% rule applied later)
INCLUDE_REPLIES = True
FETCH_ALL_REPLIES = True        # commentThreads gives only 5 inline replies; fetch the rest
COMMENT_ORDER = 'time'          # avoids YouTube's relevance ranking
MAX_VIDEO_SHARE = 0.30
TARGET_MIN_COMMENTS = 3000

RELEVANCE_KEYWORDS = [
    'education', 'reform', 'curriculum', 'syllabus', 'school', 'exam', 'o/l', 'a/l', 'grade', 'teacher',
    'module', 'vocational', 'student',
    'අධ්‍යාපන', 'ප්‍රතිසංස්කරණ', 'විෂය', 'පාසල', 'ගුරු', 'ශ්‍රේණිය', 'පෙළ', 'විභාග', 'මොඩියුල', 'සිසු',
]
SRI_LANKA_KEYWORDS = ['sri lanka', 'srilanka', 'lanka', 'ශ්‍රී ලංකා', 'ලංකා', 'harini', 'හරිනි']

COLLECTION_DATE = datetime.now(timezone.utc).isoformat()
print('Total queries:', sum(len(v) for v in SEARCH_QUERIES.values()))

Total queries: 19


### Quota-aware API wrapper

In [5]:
QUOTA_COST = {'search': 100, 'videos': 1, 'commentThreads': 1, 'comments': 1}
quota_used = Counter()

class QuotaExceeded(Exception):
    pass

def http_reason(e):
    try:
        return json.loads(e.content.decode())['error']['errors'][0]['reason']
    except Exception:
        return f'http_{getattr(e.resp, "status", "unknown")}'

def api_call(request, kind, max_retries=4):
    for attempt in range(max_retries):
        try:
            resp = request.execute()
            quota_used[kind] += QUOTA_COST[kind]
            return resp
        except HttpError as e:
            quota_used[kind] += QUOTA_COST[kind]
            reason = http_reason(e)
            if reason in ('quotaExceeded', 'dailyLimitExceeded'):
                raise QuotaExceeded(reason)
            if e.resp.status in (500, 502, 503, 504) and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise

def quota_total():
    return sum(quota_used.values())

### Search for videos (results cached to Drive)

In [6]:
SEARCH_CACHE = RAW_DIR / 'search_cache.json'
search_cache = json.loads(SEARCH_CACHE.read_text()) if SEARCH_CACHE.exists() else {}

def search_videos(query, lang):
    key = f'{lang}|{query}'
    if key in search_cache:
        return search_cache[key]
    req = youtube.search().list(
        q=query, part='snippet', type='video', maxResults=min(RESULTS_PER_QUERY, 50),
        regionCode=REGION_CODE, relevanceLanguage=lang,
        publishedAfter=PUBLISHED_AFTER, order='relevance')
    resp = api_call(req, 'search')
    rows = [{
        'query': query, 'query_lang': lang,
        'video_id': it['id']['videoId'],
        'title': it['snippet']['title'],
        'channel_id': it['snippet']['channelId'],
        'channel_title': it['snippet']['channelTitle'],
        'video_published_at': it['snippet']['publishedAt'],
        'searched_at': datetime.now(timezone.utc).isoformat(),
    } for it in resp.get('items', [])]
    search_cache[key] = rows
    SEARCH_CACHE.write_text(json.dumps(search_cache, ensure_ascii=False, indent=1))
    return rows

hits = []
for lang, queries in SEARCH_QUERIES.items():
    for q in queries:
        hits.extend(search_videos(q, lang))

search_df = pd.DataFrame(hits)
print(f'{len(search_df)} hits, {search_df.video_id.nunique()} unique videos, quota used: {quota_total()}')
search_df.groupby('query_lang').video_id.nunique()

475 hits, 306 unique videos, quota used: 1900


,video_id
query_lang,
en,197
si,121


### Video details (comment counts, descriptions)


In [7]:
cand = (search_df.groupby('video_id')
        .agg(title=('title', 'first'), channel_id=('channel_id', 'first'),
             channel_title=('channel_title', 'first'), video_published_at=('video_published_at', 'first'),
             matched_queries=('query', lambda s: ' || '.join(sorted(set(s)))),
             query_langs=('query_lang', lambda s: ','.join(sorted(set(s)))))
        .reset_index())

details = []
ids = cand.video_id.tolist()
for i in range(0, len(ids), 50):
    resp = api_call(youtube.videos().list(part='snippet,statistics', id=','.join(ids[i:i+50])), 'videos')
    for it in resp.get('items', []):
        st, sn = it.get('statistics', {}), it['snippet']
        details.append({
            'video_id': it['id'],
            'description': sn.get('description', ''),
            'tags': '|'.join(sn.get('tags', [])),
            'default_audio_language': sn.get('defaultAudioLanguage'),
            'view_count': int(st.get('viewCount', 0)),
            'comment_count': int(st['commentCount']) if 'commentCount' in st else None,  # missing = disabled
        })
videos = cand.merge(pd.DataFrame(details), on='video_id', how='left')
print('Videos with comments disabled/unavailable:', videos.comment_count.isna().sum())

Videos with comments disabled/unavailable: 11


### Relevance filter and manual review

In [8]:
def kw_hits(text, kws):
    t = str(text).lower()
    return sum(k.lower() in t for k in kws)

meta_text = videos.title.fillna('') + ' ' + videos.description.fillna('') + ' ' + videos.tags.fillna('')
videos['topic_kw_hits'] = meta_text.apply(lambda t: kw_hits(t, RELEVANCE_KEYWORDS))
videos['lk_kw_hits'] = meta_text.apply(lambda t: kw_hits(t, SRI_LANKA_KEYWORDS))
videos['local_query'] = videos.query_langs.str.contains('si')

videos['exclusion_reason'] = np.select(
    [videos.comment_count.isna(),
     videos.comment_count < MIN_COMMENT_COUNT,
     videos.topic_kw_hits == 0,
     (videos.lk_kw_hits == 0) & ~videos.local_query],
    ['comments_disabled_or_unavailable', 'too_few_comments', 'no_topic_keywords', 'not_sri_lanka'],
    default='')
videos['auto_include'] = videos.exclusion_reason == ''

# ---- Manual review: fill in after reading the titles below ----
MANUAL_EXCLUDE = {}   # {'videoId': 'reason'}
MANUAL_INCLUDE = {}   # {'videoId': 'reason'}
# ---------------------------------------------------------------
videos['include'] = videos.auto_include
for vid, why in MANUAL_EXCLUDE.items():
    videos.loc[videos.video_id == vid, ['include', 'exclusion_reason']] = [False, f'manual: {why}']
for vid, why in MANUAL_INCLUDE.items():
    videos.loc[videos.video_id == vid, ['include', 'exclusion_reason']] = [True, f'manual include: {why}']

videos.to_csv(RAW_DIR / 'videos_candidates.csv', index=False)
print(videos.exclusion_reason.replace('', 'included').value_counts(), '\n')
print('Included videos:', videos.include.sum(), '| channels:', videos.loc[videos.include, 'channel_id'].nunique())
videos.loc[videos.include, ['video_id', 'channel_title', 'title', 'comment_count', 'video_published_at']] \
      .sort_values('comment_count', ascending=False)

exclusion_reason
too_few_comments                    196
included                             72
not_sri_lanka                        18
comments_disabled_or_unavailable     11
no_topic_keywords                     9
Name: count, dtype: int64 

Included videos: 72 | channels: 55


,video_id,channel_title,title,comment_count,video_published_at
28,4ii1rkZDhgI,Health Geography Hub,Why Sri Lanka is NOT Part of India,3597.0,2024-10-01T15:30:02Z
234,h9nL4RwNAEE,Truth With Sama,ශ්‍රී ලංකාවේ හොදම පාසල් 10 😱 | Top 10 best school in the srilanka #truthwithsama #top10 #viral,2497.0,2024-05-29T10:14:08Z
75,F-DLnJM7jpg,Ada Derana,"O/L, A/L සහ ශිෂ්‍යත්ව විභාගය ගැන දැනුම්දීමක් | Ada Derana",1036.0,2026-06-24T10:37:50Z
141,RHfwHM3M9Fc,Ada Derana,අධ්‍යාපන ප්‍රතිසංස්කරණ ඇත්තටම අකුල ගත්තේ ඇයි | Ada Derana,845.0,2026-01-13T19:27:31Z
14,1mGozsjkLYQ,Maths Crew,How to do AL private | Private AL Apply | උසස් පෙළ විභාගයට private apply කරන්නෙ කොහොමද ?,761.0,2025-01-19T13:00:02Z
...,...,...,...,...,...
182,XwELPDq-JO8,UICB TELEVISION,අධ්‍යාපන ක්‍රමය 2026 ජනවාරි 1 සිට නව ප්‍රතිසංස්කරණයකට @UICBTELEVISION,17.0,2025-07-16T08:07:15Z
35,5rkNywVfrTU,WION,Sri Lanka Delays Grade 6 Reforms After Adult Website Link Found In Textbook | WION,16.0,2026-01-15T04:09:49Z
170,W32WN3WT_i8,Sri Lankan YouTube School,Familiarizing 2026 Grade 6 English New Curriculum Reforms TG,16.0,2025-11-23T12:30:44Z
268,ovjGS5qae5Y,Shakthi FM,Only 7 subjects for O/L from now on? | Massive change in Sri Lanka&#39;s education system! 🚨 Shakthi FM,16.0,2026-09-15T09:28:53Z


### Comment-fetching functions

In [9]:
def hash_author(channel_id):
    if not channel_id:
        return None
    return hashlib.sha256((AUTHOR_SALT + channel_id).encode()).hexdigest()[:16]

def make_row(c, video_id, is_reply, parent_id):
    s = c['snippet']
    return {
        'comment_id': c['id'],
        'parent_id': parent_id,
        'is_reply': is_reply,
        'video_id': video_id,
        'text_original': s.get('textOriginal', s.get('textDisplay')),
        'like_count': s.get('likeCount'),
        'comment_published_at': s.get('publishedAt'),
        'comment_updated_at': s.get('updatedAt'),
        'author_hash': hash_author(s.get('authorChannelId', {}).get('value')),  # no names stored
    }

def fetch_all_replies(parent_id, video_id):
    out, token = [], None
    while True:
        resp = api_call(youtube.comments().list(part='snippet', parentId=parent_id, maxResults=100,
                                                pageToken=token, textFormat='plainText'), 'comments')
        out += [make_row(r, video_id, True, parent_id) for r in resp.get('items', [])]
        token = resp.get('nextPageToken')
        if not token:
            return out

def fetch_video_comments(video_id, max_comments=MAX_COMMENTS_PER_VIDEO):
    rows, token = [], None
    while len(rows) < max_comments:
        try:
            resp = api_call(youtube.commentThreads().list(
                part='snippet,replies', videoId=video_id, maxResults=100, pageToken=token,
                textFormat='plainText', order=COMMENT_ORDER), 'commentThreads')
        except HttpError as e:
            return rows, http_reason(e)      # e.g. commentsDisabled
        for item in resp.get('items', []):
            top = item['snippet']['topLevelComment']
            rows.append(make_row(top, video_id, False, None))
            n_rep = item['snippet'].get('totalReplyCount', 0)
            if INCLUDE_REPLIES and n_rep:
                inline = item.get('replies', {}).get('comments', [])
                if FETCH_ALL_REPLIES and n_rep > len(inline):
                    rows += fetch_all_replies(top['id'], video_id)
                else:
                    rows += [make_row(r, video_id, True, top['id']) for r in inline]
        token = resp.get('nextPageToken')
        if not token:
            break
    return rows, 'ok'

### Collect comments (resumable)

In [10]:
selected = videos[videos.include].sort_values('comment_count', ascending=False)
status_log = {}

try:
    for _, v in selected.iterrows():
        ck = CKPT_DIR / f'{v.video_id}.json'
        if ck.exists():
            status_log[v.video_id] = json.loads(ck.read_text())['status']
            continue
        rows, status = fetch_video_comments(v.video_id)
        ck.write_text(json.dumps({'video_id': v.video_id, 'status': status,
                                  'collected_at': datetime.now(timezone.utc).isoformat(),
                                  'comments': rows}, ensure_ascii=False))
        status_log[v.video_id] = status
        print(f'{v.video_id}  {status:<18} {len(rows):>5} comments   quota used: {quota_total()}')
except QuotaExceeded:
    print('Daily quota reached; checkpoints saved. Re-run this cell tomorrow to resume.')

print('Quota used this session:', dict(quota_used), '=', quota_total())

4ii1rkZDhgI  ok                   901 comments   quota used: 1918
h9nL4RwNAEE  ok                   861 comments   quota used: 1927
F-DLnJM7jpg  ok                  1036 comments   quota used: 1950
RHfwHM3M9Fc  ok                   845 comments   quota used: 1976
1mGozsjkLYQ  ok                   769 comments   quota used: 1990
-pHFRz2p-Nw  ok                   730 comments   quota used: 2008
iKJSiiynAWQ  ok                   610 comments   quota used: 2029
WoD8P9g8OHs  ok                   554 comments   quota used: 2041
Srr-sYiIDuI  ok                   516 comments   quota used: 2050
j42KPskCsc0  ok                   456 comments   quota used: 2056
k7Y15uXnD_Y  ok                   433 comments   quota used: 2066
D9m0gq1sqcc  ok                   378 comments   quota used: 2079
JwMy5pCk6i4  ok                   300 comments   quota used: 2085
FKoS8yJOKWo  ok                   264 comments   quota used: 2087
hoUH6bmB9a8  ok                   242 comments   quota used: 2089
NKp9ZXhANA

### Save the unmodified raw file

In [15]:
records = []
for ck in sorted(CKPT_DIR.glob('*.json')):
    d = json.loads(ck.read_text())
    for r in d['comments']:
        r['collected_at'] = d['collected_at']
        records.append(r)

raw = pd.DataFrame(records)
vid_meta = videos[['video_id', 'channel_id', 'channel_title', 'title', 'video_published_at', 'matched_queries']] \
           .rename(columns={'title': 'video_title'})
raw = raw.merge(vid_meta, on='video_id', how='left')
raw.to_csv(RAW_DIR / 'raw_comments.csv', index=False, encoding='utf-8-sig')   # never edited after this
print(raw.shape)
raw.head(500)

(12793, 15)


,comment_id,parent_id,is_reply,video_id,text_original,like_count,comment_published_at,comment_updated_at,author_hash,collected_at,channel_id,channel_title,video_title,video_published_at,matched_queries
0,UgxzwPiB2TauS3ZJNNZ4AaABAg,None,False,-0pCTULWg8g,Ane Teecha Dan barida ape babawa danna,0,2026-11-04T10:39:42Z,2026-11-04T10:39:42Z,1692ab2fb2c9c7dc,2026-11-20T09:17:40.519707+00:00,UC6Mn7YVeOwxI2h6usI1sO1Q,Schoooly App-හරි පුදුම ඉස්කෝලේ,2026 නව 1 ශ්‍රේණිය පන්තිය ආරම්භය!,2025-12-31T02:31:48Z,පළමු ශ්‍රේණිය නව ප්‍රතිසංස්කරණ
1,UgyfsK0A8bV52nudHHV4AaABAg,None,False,-0pCTULWg8g,Teacher ape baba 1 sereniye teacher clz ekata sambanda karaganna puluwanda,0,2026-10-29T02:41:36Z,2026-10-29T02:41:36Z,76cd7dd46cf7faf5,2026-11-20T09:17:40.519707+00:00,UC6Mn7YVeOwxI2h6usI1sO1Q,Schoooly App-හරි පුදුම ඉස්කෝලේ,2026 නව 1 ශ්‍රේණිය පන්තිය ආරම්භය!,2025-12-31T02:31:48Z,පළමු ශ්‍රේණිය නව ප්‍රතිසංස්කරණ
2,Ugx4-snGmG-yERJVvKV4AaABAg,None,False,-0pCTULWg8g,අපේ පුතාවත් සම්බන්ධ කරන්න කැමතියි ටීචර් ඔයා හරිම ලස්සනට උගන්වනවා තෙරුවන් සරණයි ටිච,0,2026-10-11T16:30:22Z,2026-10-11T16:30:22Z,c9ff2a64dfd4901a,2026-11-20T09:17:40.519707+00:00,UC6Mn7YVeOwxI2h6usI1sO1Q,Schoooly App-හරි පුදුම ඉස්කෝලේ,2026 නව 1 ශ්‍රේණිය පන්තිය ආරම්භය!,2025-12-31T02:31:48Z,පළමු ශ්‍රේණිය නව ප්‍රතිසංස්කරණ
3,UgzThGekDYmgBZe8lTB4AaABAg,None,False,-0pCTULWg8g,මේ ළමයින්ට මොන පන්තිද,1,2026-06-17T10:17:55Z,2026-06-17T10:17:55Z,daeb4003ebc0ebb7,2026-11-20T09:17:40.519707+00:00,UC6Mn7YVeOwxI2h6usI1sO1Q,Schoooly App-හරි පුදුම ඉස්කෝලේ,2026 නව 1 ශ්‍රේණිය පන්තිය ආරම්භය!,2025-12-31T02:31:48Z,පළමු ශ්‍රේණිය නව ප්‍රතිසංස්කරණ
4,UgzHRJI41skEsejDuOl4AaABAg,None,False,-0pCTULWg8g,ටිච ක්ලාස් එකට ජොයින් වෙින්න ඔනි.කොහොමද ?,1,2026-06-10T11:02:36Z,2026-06-10T11:02:36Z,6d67588bb5466981,2026-11-20T09:17:40.519707+00:00,UC6Mn7YVeOwxI2h6usI1sO1Q,Schoooly App-හරි පුදුම ඉස්කෝලේ,2026 නව 1 ශ්‍රේණිය පන්තිය ආරම්භය!,2025-12-31T02:31:48Z,පළමු ශ්‍රේණිය නව ප්‍රතිසංස්කරණ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Ugw6ftyK1VTVkBs53Wx4AaABAg,None,False,-pHFRz2p-Nw,තෝරගන්න විෂයන් ඇතුළෙ අනිවාර්ය කරන්න ඕන ඒව තියෙනවා සිංහල ආගම අයින් කරල IT අනිවාර්ය කරන්න තිබ්බෙ,1,2025-07-19T12:59:53Z,2025-07-19T12:59:53Z,3d324362f32d8540,2026-11-20T09:17:05.223558+00:00,UC_0nRvlMjooFAZEiMTkCglA,Maths with Chathura Madushan,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න,2025-07-18T12:33:39Z,නව විෂය නිර්දේශය පාසල්
496,Ugy78Su5UDXhUqG9RQV4AaABAg,None,False,-pHFRz2p-Nw,සෞන්දර්ය විෂය ඇතුලේ රසාස්වාදය තියෙයි ද ?,1,2025-07-19T12:56:03Z,2025-07-19T12:56:03Z,d34994a7407eb263,2026-11-20T09:17:05.223558+00:00,UC_0nRvlMjooFAZEiMTkCglA,Maths with Chathura Madushan,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න,2025-07-18T12:33:39Z,නව විෂය නිර්දේශය පාසල්
497,Ugy0D6B9hTiSEVrVNAh4AaABAg,None,False,-pHFRz2p-Nw,එතකොට සර් ජී.පී එක අනුව උසස් පෙළට කරන සබ්ජෙක්ට් එක තීරණය කරන්නේ කොහොමද,3,2025-07-19T12:14:35Z,2025-07-19T12:14:35Z,16c735fb8a624663,2026-11-20T09:17:05.223558+00:00,UC_0nRvlMjooFAZEiMTkCglA,Maths with Chathura Madushan,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න,2025-07-18T12:33:39Z,නව විෂය නිර්දේශය පාසල්
498,UgzuKOQw706QREXW4Rh4AaABAg,None,False,-pHFRz2p-Nw,Ithihasaya aniwarya nokara anagathe Sri Lankika daruwanta lak ithihasaya gana watahimak Nathan aparade,2,2025-07-19T12:12:10Z,2025-07-19T12:12:10Z,7e3bea2a616480ec,2026-11-20T09:17:05.223558+00:00,UC_0nRvlMjooFAZEiMTkCglA,Maths with Chathura Madushan,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න,2025-07-18T12:33:39Z,නව විෂය නිර්දේශය පාසල්


### Language / script detection

In [12]:
SI_RE  = re.compile(r'[\u0D80-\u0DFF]')    # Sinhala block
TA_RE  = re.compile(r'[\u0B80-\u0BFF]')    # Tamil block (used only to exclude)
LAT_RE = re.compile(r'[A-Za-z]')

SINGLISH_MARKERS = {'mama', 'oya', 'api', 'eka', 'ekak', 'nam', 'neda', 'nedda', 'kiyala', 'karanna', 'wenna',
                    'thiyenne', 'thiyena', 'nathi', 'naha', 'godak', 'hari', 'ithin', 'mokada', 'wage', 'dan',
                    'oyala', 'lamai', 'lamaynta', 'ganna', 'denna', 'puluwan', 'balanna', 'kohomada', 'ne', 'ow'}

def tamil_share(text):
    t = str(text) if pd.notna(text) else ''
    n_ta = len(TA_RE.findall(t))
    total = n_ta + len(SI_RE.findall(t)) + len(LAT_RE.findall(t))
    return n_ta / total if total else 0.0

def language_group(text):
    t = str(text) if pd.notna(text) else ''
    n_si, n_lat = len(SI_RE.findall(t)), len(LAT_RE.findall(t))
    total = n_si + n_lat
    if total == 0:
        return 'no_letters (emoji/symbols/numbers)'
    si_share = n_si / total
    if 0.2 <= si_share <= 0.8:
        return 'mixed_sinhala_english'
    if si_share > 0.8:
        return 'sinhala'
    toks = re.findall(r'[a-z]+', t.lower())
    return 'romanised_sinhala (singlish)' if sum(w in SINGLISH_MARKERS for w in toks) >= 2 else 'english'

### Duplicates, Tamil exclusion, 30% rule

In [13]:
def enforce_video_cap(df, max_share=MAX_VIDEO_SHARE, seed=SEED):
    df = df.copy()
    rng = np.random.default_rng(seed)
    while True:
        counts = df.video_id.value_counts()
        cap = math.floor(max_share * len(df))
        over = counts[counts > cap]
        if over.empty:
            return df
        drop_idx = []
        for vid, n in over.items():
            idx = df.index[df.video_id == vid].to_numpy()
            drop_idx += list(rng.choice(idx, size=n - cap, replace=False))
        df = df.drop(index=drop_idx)

n0 = len(raw)
corpus = raw.drop_duplicates('comment_id').reset_index(drop=True)
n_dup_ids = n0 - len(corpus)

# Scope: Sinhala + English only. Drop comments where >= 20% of letters are Tamil script.
is_ta = corpus.text_original.apply(tamil_share) >= 0.2
n_tamil = int(is_ta.sum())
corpus = corpus[~is_ta].reset_index(drop=True)

n1 = len(corpus)
corpus = enforce_video_cap(corpus).reset_index(drop=True)
n_capped = n1 - len(corpus)

corpus.to_csv(RAW_DIR / 'corpus_selected_raw_text.csv', index=False, encoding='utf-8-sig')

share = corpus.video_id.value_counts(normalize=True)
checks = {
    'raw rows': n0,
    'duplicate comment IDs removed': n_dup_ids,
    'Tamil-script comments removed (out of scope)': n_tamil,
    'removed by 30% cap': n_capped,
    'final unique comments': len(corpus),
    'videos': corpus.video_id.nunique(),
    'channels': corpus.channel_id.nunique(),
    'largest video share': f'{share.max():.1%}',
    'meets >=3000 comments': len(corpus) >= TARGET_MIN_COMMENTS,
    'meets >=3 channels': corpus.channel_id.nunique() >= 3,
    'meets <=30% per video': bool(share.max() <= MAX_VIDEO_SHARE + 1e-9),
}
pd.Series(checks, name='value').to_frame()

,value
raw rows,12793
duplicate comment IDs removed,41
Tamil-script comments removed (out of scope),144
removed by 30% cap,0
final unique comments,12608
videos,72
channels,55
largest video share,8.2%
meets >=3000 comments,True
meets >=3 channels,True


### Collection log (for reproducibility)

In [14]:
log = {
    'collection_date_utc': COLLECTION_DATE,
    'api': 'YouTube Data API v3',
    'language_scope': 'Sinhala and English (incl. romanised Sinhala); Tamil-script comments excluded',
    'search_queries': SEARCH_QUERIES,
    'params': {k: globals()[k] for k in ['PUBLISHED_AFTER', 'REGION_CODE', 'RESULTS_PER_QUERY', 'MIN_COMMENT_COUNT',
                                         'MAX_COMMENTS_PER_VIDEO', 'INCLUDE_REPLIES', 'FETCH_ALL_REPLIES',
                                         'COMMENT_ORDER', 'MAX_VIDEO_SHARE', 'SEED']},
    'manual_exclude': MANUAL_EXCLUDE, 'manual_include': MANUAL_INCLUDE,
    'video_status': status_log,
    'quota_used_this_session': dict(quota_used),
    'counts': {k: str(v) for k, v in checks.items()},
}
(RAW_DIR / 'collection_log.json').write_text(json.dumps(log, ensure_ascii=False, indent=2, default=str))
print('Saved:', *[p.name for p in RAW_DIR.iterdir() if p.is_file()], sep='\n  ')

Saved:
  search_cache.json
  videos_candidates.csv
  raw_comments.csv
  corpus_selected_raw_text.csv
  collection_log.json
